In [1]:
import torch
import matplotlib.pyplot as plt
import numpy as np

from pytorch_ood.detector import KNN, Mahalanobis, GMM
from pytorch_ood.utils import OODMetrics

In [2]:
import torch


medsam_zs_shifts={
    "id": "MedSAM_zs_ZGT_masses",
    "near": "medSAM_zs_INbreast_masses",
    "far": "medSAM_zs_MamaMia_masses",
}


medsam_lora_shifts={
    "id": "medSAM_LoRA_ZGT_allmasses",
    "near": "medSAM_LoRA_INbreast_allmasses",
    "far": "medSAM_LoRA_MamaMia_masses",
}


versamammo_shifts = {
    "id": "VersaMammo_ZGT_allmasses",
    "near": "VersaMammo_INbreast_allmasses",
    "far": "VersaMammo_MamaMia_masses",
}

mammofm_shifts = {
    "id": "MammoFM_ZGT_allmasses",
    "near": "MammoFM_INbreast_allmasses",
    "far": "MammoFM_MamaMia_masses",
}

detector_classes = {
    "Mahalanobis": Mahalanobis,
    "KNN": KNN,
    "GMM": GMM,
}

results = {
    "MedSAM_zs":{},
    "MedSAM_LoRA":{},
    "VersaMammo": {},
    "MammoFM": {},
}

model_shifts = {
    "MedSAM_zs":medsam_zs_shifts,
    "MedSAM_LoRA":medsam_lora_shifts,
    "VersaMammo": versamammo_shifts,
    "MammoFM": mammofm_shifts,
}

for model_name, shifts in model_shifts.items():
    # Load each feature set once for this model
    Z_id = torch.load(shifts["id"], map_location="cpu")
    Z_near_ood = torch.load(shifts["near"], map_location="cpu")
    Z_far_ood = torch.load(shifts["far"], map_location="cpu")

    Y_id = torch.zeros(Z_id.shape[0], dtype=torch.long)

    for detector_name, DetectorCls in detector_classes.items():
        det = DetectorCls(encoder=None)
        det.fit_features(Z_id, Y_id)

        scores_near_ood = det.predict_features(Z_near_ood)
        scores_far_ood = det.predict_features(Z_far_ood)

        # Ensure outputs are tensors, since some detectors may return NumPy arrays
        scores_near_ood = torch.as_tensor(
            scores_near_ood,
            dtype=torch.float32,
        ).flatten()

        scores_far_ood = torch.as_tensor(
            scores_far_ood,
            dtype=torch.float32,
        ).flatten()

        # Joint normalization makes near/far distances comparable
        all_scores = torch.cat(
            [scores_near_ood, scores_far_ood],
            dim=0,
        )

        score_min = all_scores.min()
        score_max = all_scores.max()
        score_range = score_max - score_min

        if score_range.item() > 0:
            scores_near_ood_norm = (
                scores_near_ood - score_min
            ) / score_range

            scores_far_ood_norm = (
                scores_far_ood - score_min
            ) / score_range
        else:
            scores_near_ood_norm = torch.zeros_like(scores_near_ood)
            scores_far_ood_norm = torch.zeros_like(scores_far_ood)

        results[model_name][detector_name] = {
            "near_avg": scores_near_ood_norm.mean().item(),
            "far_avg": scores_far_ood_norm.mean().item(),
        }

In [3]:
from pprint import pprint

pprint(results)

{'MammoFM': {'GMM': {'far_avg': 0.381621390581131,
                     'near_avg': 0.313480943441391},
             'KNN': {'far_avg': 0.4987492263317108,
                     'near_avg': 0.4348039925098419},
             'Mahalanobis': {'far_avg': 0.4024248719215393,
                             'near_avg': 0.4870016276836395}},
 'MedSAM_LoRA': {'GMM': {'far_avg': 0.047175440937280655,
                         'near_avg': 0.5217790007591248},
                 'KNN': {'far_avg': 0.18456804752349854,
                         'near_avg': 0.7240281701087952},
                 'Mahalanobis': {'far_avg': 0.03770052641630173,
                                 'near_avg': 0.49711230397224426}},
 'MedSAM_zs': {'GMM': {'far_avg': 0.02115505002439022,
                       'near_avg': 0.3798784911632538},
               'KNN': {'far_avg': 0.11584600061178207,
                       'near_avg': 0.6487478613853455},
               'Mahalanobis': {'far_avg': 0.026678085327148438,
                 

In [4]:
import json

with open("ood_average_normalized_distances.json", "w") as f:
    json.dump(results, f, indent=4)